# Reproducibility, ethics và portfolio

## Mục tiêu
- Lưu model pipeline cùng metadata tối thiểu.
- Thiết kế contract giữa training và prediction.
- Nhận diện privacy, fairness, drift và giới hạn portfolio.

In [ ]:
from pathlib import Path
import joblib
from sklearn.datasets import load_iris
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

repo_root = Path('.') if Path('data').exists() else Path('..')
iris = load_iris(as_frame=True)
pipeline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1_000))
pipeline.fit(iris.data, iris.target)
artifact_path = repo_root / 'artifacts/iris_pipeline.joblib'
artifact_path.parent.mkdir(exist_ok=True)
joblib.dump({'model': pipeline, 'feature_names': iris.feature_names, 'version': '0.1.0'}, artifact_path)
artifact_path.exists()

## Model contract
Một artifact chỉ hữu ích khi input schema rõ: tên feature, type, đơn vị, range hợp lý, missing-value rule và prediction version. Validation ở boundary nên trả lỗi rõ ràng thay vì âm thầm đổi schema. Production còn cần access control, log an toàn và rollback plan.

In [ ]:
artifact = joblib.load(artifact_path)
expected = artifact['feature_names']
def validate_input(frame):
    missing = set(expected) - set(frame.columns)
    if missing:
        raise ValueError(f'thiếu feature: {sorted(missing)}')
    return frame.loc[:, expected]

validated = validate_input(iris.data.head(2))
artifact['model'].predict(validated)

## Ethics và monitoring checklist
- **Privacy:** có quyền dùng data? có thật sự cần PII không? có retention/deletion policy?
- **Fairness:** error rate và benefit có khác bất lợi giữa các nhóm có liên quan không?
- **Drift:** input distribution, target rate, performance khi label về muộn thay đổi ra sao?
- **Human oversight:** ai chịu trách nhiệm khi prediction sai, người dùng có thể khiếu nại không?
- **Security:** model/artifact và data không được public một cách vô tình.

Không triển khai mô hình cho quyết định high-impact (tín dụng, tuyển dụng, y tế...) chỉ dựa vào checklist này.

## Portfolio có sức thuyết phục
Mỗi project nên trả lời: vấn đề gì, ai dùng kết quả, data từ đâu/có license không, quyết định cleaning nào, baseline/metric nào, kết quả/limitation ra sao và hành động nào được đề xuất. README ngắn có link notebook sạch thường tốt hơn nhiều notebook không có câu chuyện.

## Tự luyện
1. Bỏ một cột khỏi `iris.data` rồi gọi `validate_input`; đọc message lỗi.
2. Thêm metadata `trained_at`, random seed, data version vào artifact.
3. Phác thảo project portfolio dùng một dataset công khai và nêu privacy risk.

**Checkpoint:** Bạn không commit artifact/data thật vào repo công khai nếu chưa xác định quyền chia sẻ.